In [15]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

## Part 1. Loading the dataset

In [16]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
data = pd.read_csv('science_data_large.csv')

# Display the first 15 rows of the dataset
print(data.head(15))

# Display a summary of the dataset information
print(data.info())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## Part 2. Splitting the dataset

In [17]:
# Take the pandas dataset and split it into our features (X) and label (y)
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X = data[['Mols KCL', 'Temperature °C']]
y = data['Size nm^3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

print(X_train.head())
print(y_train.head())

     Mols KCL  Temperature °C
662       646             775
12         70             206
189       385             705
7         869             143
145       649             475
662    1.022523e+06
12     3.145200e+04
189    5.555450e+05
7      2.718260e+05
145    6.342843e+05
Name: Size nm^3, dtype: float64


## Part 3. Perform a Linear Regression

In [18]:
# Use sklearn to train a model on the training set
linear_model = LinearRegression().fit(X_train, y_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample = pd.DataFrame([[0.5, 37]], columns=['Mols KCL', 'Temperature °C'])
size_prediction = linear_model.predict(sample)
print(f"Sample prediction: {size_prediction}")
# Report on the score for that model, in your own words (markdown, not code) explain what the score means

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX

y_prediction = linear_model.predict(X_test)
r_2_score = r2_score(y_test, y_prediction)
print(f"R-squared score of the model: {r_2_score}")
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': linear_model.coef_
})
intercept = linear_model.intercept_
temperature_coefficient = coefficients[coefficients['Feature'] == 'Temperature °C']['Coefficient'].values[0]
mols_coefficient = coefficients[coefficients['Feature'] == 'Mols KCL']['Coefficient'].values[0]

print(f"Temperature Coefficient: {temperature_coefficient}")
print(f"Mols KCL Coefficient: {mols_coefficient}")
print(f"Intercept: {intercept}")

Sample prediction: [-378480.96484035]
R-squared score of the model: 0.8908241194929963
Temperature Coefficient: 873.6797684817209
Mols KCL Coefficient: 1019.9524742427762
Intercept: -411317.0925112958


the score for the linear regression model tells us how well the model explains variance in the target variable, which is the size of the slime (nm^3)
a score close to 1 means the model explains most of the variance in the data, meaning its a good fit for predicting the size based on input
a score close to 0 means the model does not explain much abt variance, but there could be factors or relationships affecting the size that the model cant capture

Equation: $h(x) = 1034.4987 \times \text{Mols KCL} + 884.7876 \times \text{Temperature °C} - 422873.4831$


Sample equation: $E = mc^2$

## Part 4. Use Cross Validation

In [19]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
cv_scores = cross_val_score(linear_model, X, y, cv=10)
# Report on their finding and their significance

# Output the cross-validation scores and their average
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean cross-validation score: {cv_scores.mean()}")

Cross-validation scores: [0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397
 0.87941022 0.86349411 0.78353682 0.88686516]
Mean cross-validation score: 0.8552450341984701


The cross-validation scores range from 0.78 to 0.89, with a mean score of 0.86. This indicates that the model performs consistently across different subsets of the data and generalizes well. The relatively small variation in the scores suggests that the model is not overfitting and is reliable for making predictions. Overall, the model explains around 85.5% of the variance, which shows strong predictive performance

## Part 5. Using Polynomial Regression

In [20]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2

# Report on the metrics and output the resultant equation as you did in Part 3.

poly = PolynomialFeatures(degree=2)

poly_X_train = poly.fit_transform(X_train)
poly_X_test = poly.transform(X_test)

poly_model = LinearRegression().fit(poly_X_train, y_train)

poly_y_prediction = poly_model.predict(poly_X_test)

poly_r_2_score = r2_score(y_test, poly_y_prediction)
print(f"Polynomial R-squared score: {poly_r_2_score}")

coefficients_poly = pd.DataFrame({
    'Feature': poly.get_feature_names_out(X.columns),
    'Coefficient': poly_model.coef_
})

print(coefficients_poly)

poly_intercept = poly_model.intercept_

Polynomial R-squared score: 1.0
                   Feature   Coefficient
0                        1  0.000000e+00
1                 Mols KCL -9.704076e-08
2           Temperature °C  1.200000e+01
3               Mols KCL^2  2.857143e-02
4  Mols KCL Temperature °C  2.000000e+00
5         Temperature °C^2 -1.413714e-11


the score for the polynomial regression model is 1 which indicates a perfect fit of the model to the data. This means that the polynomial model is able to explain all the variance in the target variable; the size of the slime

The coefficient of each polynomial feature shows how each term in the polymonial equation contributes to predicting slime size.

Equation: $h(x) = -1.384155 \times 10^{-7} \times \text{Mols KCL} + 12.000 \times \text{Temperature °C} + 0.02857143 \times (\text{Mols KCL})^2 + 2.000 \times (\text{Mols KCL} \times \text{Temperature °C}) - 1.118466 \times 10^{-11} \times (\text{Temperature °C})^2$
